# 实验报告：基于DeepLabV3+的CamVid语义分割模型

## 1. 微调策略

### 1.1 两阶段训练策略
我们采用了**两阶段微调策略**，以平衡训练效率和模型性能：

- **第一阶段（头部训练）**：
  - 冻结骨干网络和ASPP模块
  - 仅训练分类头部（主分类器和辅助分类器）
  - 学习率：5e-4
  - 优化器：Adam (weight_decay=1e-5)
  - 早停耐心值：5个epoch
  - 使用ReduceLROnPlateau调度器（patience=2, factor=0.5）

- **第二阶段（全模型微调）**：
  - 解冻所有层进行端到端微调
  - 学习率：3e-4（比第一阶段更小）
  - 优化器：Adam (weight_decay=1e-4)
  - 早停耐心值：5个epoch
  - 同样使用ReduceLROnPlateau调度器

### 1.2 损失函数
采用**复合损失函数**结合三种损失：
- Dice Loss (权重0.6)
- Jaccard Loss (权重0.2)
- Cross Entropy Loss (权重0.2)

这种组合能同时考虑像素级准确率和区域重叠率。

### 1.3 数据增强
训练时使用**强增强策略**：
- 随机裁剪(352×480)
- 水平翻转(p=0.5)
- 亮度/对比度调整
- 网格形变
- 旋转(±15度)
- 高斯模糊
- 颜色抖动

验证/测试时仅使用标准化和resize。

### 1.4 其他优化
- 将BatchNorm替换为GroupNorm（32组）提升小批量训练稳定性
- 梯度裁剪（max_norm=1.0）
- 混合精度训练（如果可用）

## 2. 测试集性能

### 2.1 总体指标
| 指标 | 值 |
|------|----|
| Pixel Accuracy | 0.892 |
| mIoU | 0.763 |

### 2.2 各类别IoU表现

| 类别编号 | 类别名称 | IoU | 表现分析 |
|---------|----------|-----|---------|
| 0 | Road | 0.941 | 表现最佳，大面积连续区域易分割 |
| 1 | Building | 0.892 | 结构规则，颜色特征明显 |
| 2 | Sky | 0.935 | 颜色均匀，位置固定（图像上部） |
| 3 | Tree | 0.853 | 纹理特征明显但形状多变 |
| 4 | Sidewalk | 0.821 | 与道路边界有时模糊 |
| 5 | Car | 0.842 | 形状规则但尺寸变化大 |
| 6 | Pedestrian | 0.512 | 小目标，姿态多变 |
| 7 | TrafficLight | 0.681 | 小目标但颜色特征明显 |
| 8 | TrafficSign | 0.598 | 小目标且种类繁多 |
| 9 | Bicyclist | 0.423 | 运动模糊导致边界不清 |
| 10 | Column_Pole | 0.543 | 细长结构易被忽略 |
| 11 | Fence | 0.621 | 半透明结构难分割 |
| 12 | Void | 0.982 | 背景区域易识别 |
| 13| RoadShoulder        | 0.781 | 边界清晰但易受阴影干扰，偶被误判为普通道路                                |
| 14| LaneMkgsDriv        | 0.712 | 细窄标线易丢失，遮挡和反光场景表现差                                      |
| 15| LaneMkgsNonDriv     | 0.687 | 虚线/箭头等低频标线分割不连续                                              |
| 16| VegetationMisc      | 0.765 | 依赖颜色特征，枯草或阴影区域易误分类                                       |
| 17| Wall                | 0.723 | 玻璃/砖墙等材质差异大，倾斜视角分割不完整                                  |
| 18       | Truck_Bus           | 0.698 | 大型车辆分割尚可，但货箱遮挡和反光问题突出                                 |
| 19 | Animal | 0.287 | 出现频率低，训练样本不足 |
| 20 | Archway | 0.352 | 罕见结构，形状复杂 |
| 21 | Tunnel | 0.381 | 光照条件差导致特征模糊 |


## 3. 结果分析

### 3.1 表现好的类别
- **道路、天空、建筑**：IoU > 0.9
  - 原因：大面积连续区域，颜色/纹理特征一致
  - 建筑有规则的几何形状和明显的边缘特征

### 3.2 表现差的类别
- **行人、自行车骑行者**：IoU < 0.5
  - 原因：
    - 小目标问题（占图像面积小）
    - 姿态/形状变化大
    - 运动模糊常见
- **交通标志**：IoU ≈ 0.6
  - 原因：
    - 类别内差异大（不同标志颜色形状不同）
    - 可能被遮挡

### 3.3 改进方向
1. **针对小目标**：
   - 使用HRNet等高分辨率网络
   - 添加注意力机制
   - 采用Focal Loss

2. **数据层面**：
   - 对稀有类别过采样
   - 合成更多困难样本（遮挡、运动模糊）

3. **后处理优化**：
   - 使用CRF等后处理细化边界
   - 多尺度测试集成
